- run on max010
- conda env: scib_metrics

In [ ]:
# computed on donor_id and cell_type

In [ ]:
import scanpy as sc
import anndata as ad
import scib
import scib_metrics
import numpy as np
import pandas as pd

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=ad.OldFormatWarning)

In [ ]:
scenarios = ['integration_hlca', 'noIntegration_hlca', 'naiveIntegration_donor_id_hlca', 'naiveIntegration_assay_hlca', 'naiveIntegration_dataset_hlca']

In [ ]:
# use all the "fast" metrics
np.random.seed(61)

# Collect computed scores, nested dict is simple to convert to pd.DataFrame
score_dict = {}
for scenario in scenarios:
    # Initialize nested dict
    score_dict[scenario] = {}
    
    adata = ad.read_h5ad('./../../embeddings/{}.embedding.h5ad'.format(scenario))
    adata.obsm['embedding'] = adata.X

    # Cast types to string, otherwise "silent" but in scib_metrics silhoeutte computation! 
    adata.obs['cell_type_categorical'] = adata.obs['cell_type']
    adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)
    adata.obs['dataset_categorical'] = adata.obs['dataset']
    adata.obs['dataset'] = adata.obs['dataset'].astype(str)
    
    sc.pp.neighbors(adata, use_rep='embedding')

    # Compute scores
    ## Level of evaluation: batch/sample
    ### asw_batch
    score = scib_metrics.metrics.silhouette_batch(
        adata.obsm['embedding'],
        adata.obs['cell_type'],
        adata.obs['dataset']
    )
    score_dict[scenario]['asw_batch'] = score

    score = scib_metrics.metrics.silhouette_batch(
        adata.obsm['embedding'],
        adata.obs['cell_type'],
        adata.obs['dataset'],
        metric='cosine'
    )
    score_dict[scenario]['asw_batch_cosine'] = score
    
    ### bras aka. asw_batch_mean_other
    score = scib_metrics.metrics.bras(
        adata.obsm['embedding'],
        adata.obs['cell_type'],
        adata.obs['dataset']
    )
    score_dict[scenario]['bras'] = score

    score = scib_metrics.metrics.bras(
        adata.obsm['embedding'],
        adata.obs['cell_type'],
        adata.obs['dataset'],
        metric='euclidean'
    )
    score_dict[scenario]['bras_euclidean'] = score

    # bras_furthest aka. asw_batch_furthest
    score = scib_metrics.metrics.bras(
        adata.obsm['embedding'],
        adata.obs['cell_type'],
        adata.obs['dataset'],
        between_cluster_distances='furthest'
        
    )
    score_dict[scenario]['bras_furthest'] = score

    score = scib_metrics.metrics.bras(
        adata.obsm['embedding'],
        adata.obs['cell_type'],
        adata.obs['dataset'],
        between_cluster_distances='furthest',
        metric='euclidean'
    )
    score_dict[scenario]['bras_furthest_euclidean'] = score

    ### asw_label
    score = scib_metrics.metrics.silhouette_label(
        adata.obsm['embedding'],
        adata.obs['cell_type']
    )
    score_dict[scenario]['asw_label'] = score

    score = scib_metrics.metrics.silhouette_label(
        adata.obsm['embedding'],
        adata.obs['cell_type'],
        metric='cosine'
    )
    score_dict[scenario]['asw_label_cosine'] = score

    ### graph iLISI and cLISI on variable batch
    score_dict[scenario]['iLISI_batch'], score_dict[scenario]['cLISI_full'] =  scib.me.lisi.lisi_graph(adata, batch_key='dataset_categorical', label_key='cell_type_categorical', type_='knn')

    ## CiLISI
    means = []
    total = 0
    for cell_type in adata.obs['cell_type_categorical'].unique():
        tmp_adata = adata[adata.obs['cell_type_categorical']==cell_type]
        cell_type_iLISI = scib.metrics.ilisi_graph(tmp_adata, batch_key='dataset_categorical', type_='knn')
        means += [cell_type_iLISI * tmp_adata.shape[0]]
        total += tmp_adata.shape[0]
        print(cell_type, cell_type_iLISI)
    print(means)
    print(np.nansum(means)/total)
    score_dict[scenario]['CiLISI_batch'] = np.nansum(means)/total
    
    ### nmi and ari  
    neigh_result_90 = scib_metrics.nearest_neighbors.pynndescent(
                    adata.obsm['embedding'], n_neighbors=90, random_state=0)# , n_jobs=self._n_jobs)
    nmi_ari_dict = scib_metrics.nmi_ari_cluster_labels_leiden(
        neigh_result_90,
        adata.obs['cell_type'],
    )

    score_dict[scenario]['nmi'], score_dict[scenario]['ari'] = nmi_ari_dict['nmi'], nmi_ari_dict['ari']

In [ ]:
scores = pd.DataFrame(score_dict)

In [ ]:
scores

In [ ]:
pd.DataFrame(score_dict).to_csv("./../../evaluation/batch_removal_scores_real_data_hlca.csv", index=True)